# 02：工作记忆裁剪与滚动摘要
对相同素材比较：全部历史、临时裁剪、摘要＋近期消息。
本地 LangChain 1.5.0 的 `SummarizationMiddleware` 在 `before_model` 更新 State 消息；
它不是只修改临时请求。原始资料应另存，不能依赖最新 State 保留全文。


In [ ]:
import os
import json
from pathlib import Path
from uuid import uuid4
from context_lab import (
    initialize, DemoModel, live_model, dumps, estimate_messages, request_cost,
    WorkingState, Context, archive, read_archive, summary_middleware,
    demo_summary, valid_tool_pairs, recent_messages, SUMMARY_PROMPT,
)
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

DATA = initialize()
# 默认不调用外部模型。可在启动内核前设置 CH09_RUN_LIVE=1，或在此改为 True。
RUN_LIVE = os.getenv('CH09_RUN_LIVE', '0') == '1'
print('真实模型演示：', RUN_LIVE)


## 1. 固定历史：重要信息在前，最新更正在后
正文中的填充段落没有业务意义；离线摘要器只识别本章列出的标记，便于验证管线。


In [ ]:
history = [
    HumanMessage('目标：登录功能，REQ-009；演示使用 SQLite；发送频率尚未确认。', id='h-first'),
    AIMessage('先考虑密码登录。', id='a-first'),
]
for i in range(6):
    history += [HumanMessage(('布局讨论，不增加业务约定。' * 50) + str(i), id=f'h-{i}'),
                AIMessage('继续核对页面信息。', id=f'a-{i}')]
history += [HumanMessage('更新决定：使用手机号验证码，密码登录暂缓。', id='h-correction'),
            AIMessage('按新决定继续。', id='a-correction'),
            HumanMessage('现在还有哪些未确认事项？', id='h-current')]
ctx = Context('alice', 'login-lab')
thread_id = 'summary-' + uuid4().hex
archive(DATA, ctx, thread_id, history)
print('全部历史近似 token：', estimate_messages(history))


In [ ]:
trim_seen = []
@wrap_model_call
def trim_for_this_call(request, handler):
    trimmed = recent_messages(request.messages, 3)
    trim_seen.append(trimmed)
    return handler(request.override(messages=trimmed))

trim_agent = create_agent(DemoModel(), middleware=[trim_for_this_call], checkpointer=InMemorySaver())
trim_config = {'configurable': {'thread_id': 'trim-' + uuid4().hex}}
trim_result = trim_agent.invoke({'messages': history}, trim_config)
assert len(trim_result['messages']) == len(history) + 1
assert len(trim_seen[-1]) < len(history)
assert not any('REQ-009' in str(m.content) for m in trim_seen[-1])
print('临时裁剪输入估计：', estimate_messages(trim_seen[-1]))
print('State 仍有完整历史：', len(trim_result['messages']))


## 2. 摘要中间件：控制触发与观察状态替换
低阈值只用于本实验。`keep` 数的是消息，不是对话轮数；近期工具消息可能改变保留边界。
摘要模型也有输入限制；本例素材受控，不能把无限大文件交给摘要模型。


In [ ]:
summary_seen = []
@wrap_model_call
def observe_summary(request, handler):
    summary_seen.append(list(request.messages))
    return handler(request)

summary_agent = create_agent(DemoModel(), middleware=[
    summary_middleware(trigger=1800, keep=3), observe_summary], checkpointer=InMemorySaver())
summary_config = {'configurable': {'thread_id': thread_id}}
compressed = summary_agent.invoke({'messages': history}, summary_config)
sent = summary_seen[-1]
summary_messages = [m for m in sent if m.additional_kwargs.get('lc_source') == 'summarization']
assert len(summary_messages) == 1
summary_text = str(summary_messages[0].content)
for required in ['REQ-009', 'SQLite', '发送频率尚未确认']:
    assert required in summary_text
assert any('密码登录暂缓' in str(m.content) for m in sent)
assert sent[-1].id == 'h-current'
assert len(compressed['messages']) < len(history)
assert estimate_messages(sent) < estimate_messages(history)
assert len(read_archive(DATA, ctx, thread_id)) == len(history)
print('摘要后的输入估计：', estimate_messages(sent))
print(summary_text)


## 3. 工具调用与结果必须保持配对
人为令裁剪点落在工具结果上；安全裁剪从所属用户轮次开始。
再次触发压缩，验证旧摘要可以滚动合并。


In [ ]:
tool_history = [HumanMessage('读取文件'),
    AIMessage('', tool_calls=[{'name': 'read', 'args': {}, 'id': 'c-1', 'type': 'tool_call'}]),
    ToolMessage('文件内容', tool_call_id='c-1'), AIMessage('读取完成')]
assert not valid_tool_pairs(tool_history[-2:])
assert valid_tool_pairs(recent_messages(tool_history, 2))
assert valid_tool_pairs(sent)
# 真正经过 SummarizationMiddleware：保留边界落在工具结果处，仍应保住配对。
paired_agent = create_agent(DemoModel(), middleware=[summary_middleware(trigger=100, keep=2)])
paired = paired_agent.invoke({'messages': history[:-3] + tool_history})
assert valid_tool_pairs(paired['messages'])
assert any(isinstance(m, ToolMessage) for m in paired['messages'])
follow_up = [HumanMessage('无新增决定的讨论。' * 300), AIMessage('继续。'),
             HumanMessage('REQ-009 的下一步是什么？')]
rolled = summary_agent.invoke({'messages': follow_up}, summary_config)
assert len([m for m in rolled['messages'] if m.additional_kwargs.get('lc_source') == 'summarization']) == 1
assert 'SQLite' in str(rolled['messages'][0].content)
print('通过：临时裁剪不改历史，摘要改 State，工具配对、滚动保留关键事实。')


## 4. 摘要失败时，不提交替换结果
中间件通过公开的 callable summarizer 接口模拟失败。输入已在原始档案，State 的旧消息不能被成功摘要替换。


In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
def broken_summary(messages):
    raise RuntimeError('模拟摘要服务失败')
bad = create_agent(DemoModel(), middleware=[SummarizationMiddleware(
    model=DemoModel(), summarizer=broken_summary, trigger=('messages', 4), keep=('messages', 2))],
    checkpointer=InMemorySaver())
bad_config = {'configurable': {'thread_id': 'failed-' + uuid4().hex}}
try:
    bad.invoke({'messages': history}, bad_config)
except RuntimeError as exc:
    assert '模拟摘要服务失败' in str(exc)
else:
    raise AssertionError('应报告摘要失败')
assert len(bad.get_state(bad_config).values['messages']) == len(history)


## 5. 可选真实摘要与后续回答
只验证结构和关键标识，再人工检查“更正、未确认事项”有没有丢失。
若断言失败，应查看摘要输入与模型输出，而不是删除断言。


In [ ]:
if RUN_LIVE:
    real_summary = create_agent(live_model(), middleware=[
        summary_middleware(model=live_model(), trigger=1800, keep=3)])
    actual = real_summary.invoke({'messages': history})
    actual_summaries = [m for m in actual['messages'] if m.additional_kwargs.get('lc_source') == 'summarization']
    assert actual_summaries and 'REQ-009' in str(actual_summaries[0].content)
    assert 'SQLite' in str(actual_summaries[0].content)
    print(actual_summaries[0].content)
    print(actual['messages'][-1].content)
else:
    print('未执行真实摘要和真实回答。')
